# ML Zoomcamp 2026 – Homework 3: Lead Scoring (Logistic Regression)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score, accuracy_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

## Dataset

In [ ]:
url = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'
df = pd.read_csv(url)
df.head()

## Data preparation

In [ ]:
print(df.isnull().sum())

categorical = df.select_dtypes(include=['object', 'string']).columns.tolist()
numerical = [c for c in df.select_dtypes(exclude=['object', 'string']).columns if c != 'converted']
print('categorical:', categorical)
print('numerical:', numerical)

df[categorical] = df[categorical].fillna('NA')
df[numerical] = df[numerical].fillna(0.0)
df.isnull().sum().sum()

## Question 1 – mode of `industry`

In [ ]:
print(df['industry'].value_counts())
print('Answer:', df['industry'].mode()[0])

## Question 2 – correlation matrix

In [ ]:
corr = df[numerical].corr()
display(corr.round(4))

pairs = [('interaction_count', 'lead_score'),
         ('number_of_courses_viewed', 'lead_score'),
         ('number_of_courses_viewed', 'interaction_count'),
         ('annual_income', 'interaction_count')]
for a, b in pairs:
    print(f'{a} & {b}: {corr.loc[a, b]:.4f}')

## Split the data

In [ ]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

y_train = df_train.converted.values
y_val = df_val.converted.values

df_train = df_train.drop(columns='converted')
df_val = df_val.drop(columns='converted')
len(df_train), len(df_val), len(df_test)

## Question 3 – mutual information

In [ ]:
mi = {c: round(mutual_info_score(df_train[c], y_train), 2) for c in categorical}
pd.Series(mi).sort_values(ascending=False)

## Question 4 – logistic regression

In [ ]:
features = categorical + numerical

def fit_eval(feats, C=1.0):
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(df_train[feats].to_dict(orient='records'))
    X_val = dv.transform(df_val[feats].to_dict(orient='records'))
    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    return accuracy_score(y_val, model.predict(X_val))

base = fit_eval(features)
print('accuracy:', base, '-> rounded:', round(base, 2))

## Question 5 – feature elimination

In [ ]:
rows = []
for f in ['lead_source', 'number_of_courses_viewed', 'interaction_count']:
    acc = fit_eval([x for x in features if x != f])
    rows.append({'feature': f, 'accuracy': acc, 'difference': base - acc})
pd.DataFrame(rows).sort_values('difference')

## Question 6 – regularization

In [ ]:
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    print(C, round(fit_eval(features, C=C), 3))

## Answers

| Q | Answer |
|---|---|
| 1 | `technology` |
| 2 | `interaction_count` and `lead_score` |
| 3 | `lead_source` |
| 4 | 0.65 |
| 5 | `number_of_courses_viewed` |
| 6 | 0.001 |